# Baseline Models Using Pretrained Sentence Embeddings

This notebook implements **baseline classification models** using **pretrained sentence embeddings** (`all-MiniLM-L6-v2`) for Yelp reviews.

### Tasks:
1. **Sentiment classification**: Negative (1–2 stars), Neutral (3 stars), Positive (4–5 stars)  
2. **Star rating prediction**: 1–5 stars

### Models included:
- Logistic Regression
- Random Forest
- Support Vector Classifier

### Features:
- Embeddings are generated from `all-MiniLM-L6-v2`
- Stratified train/test split
- Evaluation metrics: Accuracy, Macro-F1, and per-class classification reports


In [1]:
import glob
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sentence_transformers import SentenceTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from mord import LogisticAT

In [2]:
# Load data
folder = "../Preprocessing-FeatureExtraction/cleaned-data/"
csv_files = glob.glob(os.path.join(folder, "*.csv"))
dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    df['state'] = os.path.splitext(os.path.basename(file))[0] 
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
df = df.dropna()
df['sentiment'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))
print(f"Loaded {len(df)} reviews from {len(dfs)} states.")

Loaded 5222860 reviews from 20 states.


In [3]:
# Encode text with pretrained embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")
X_embedded = embedder.encode(df['clean_text'].tolist(), batch_size=128, show_progress_bar=True)
X = np.array(X_embedded)
y_sent = df['sentiment'].values
y_star = df['stars'].values

Batches:   0%|          | 0/40804 [00:00<?, ?it/s]

In [ ]:
# Train/test split
X_train, X_test, y_sent_train, y_sent_test = train_test_split(X, y_sent, test_size=0.1, random_state=42, stratify=y_sent)
_, _, y_star_train, y_star_test = train_test_split(X, y_star, test_size=0.1, random_state=42, stratify=y_star)

In [ ]:
def train_baseline_models(X_train, y_train, X_test, y_test):    
    models = {
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Naive Bayes': MultinomialNB(),
            'SVM': SVC(kernel='linear', max_iter=2000),
            'Random Forest': RandomForestClassifier(n_estimators=200, max_features=100, max_depth=25),
            'Ordinal Logistic Regression': LogisticAT(max_iter=2000)
        }
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        
        # Per-class metrics
        report = classification_report(y_test, y_pred, output_dict=True)
        
        results[name] = {
            'model': model,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'predictions': y_pred,
            'classification_report': report
        }
        
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Macro-F1: {macro_f1:.4f}")
    
    return results


In [ ]:
# Sentiment baseline
print("\n--- Sentiment Baseline ---")
sent_model_results = train_baseline_models(X_train, y_sent_train, X_test, y_sent_test)

# Star baseline
print("\n--- Star Baseline (1-5) ---")
star_model_results = train_baseline_models(X_train, y_star_train, X_test, y_star_test)